# Detección de oportunidades — 07

**Objetivo:** clasificar las nuevas publicaciones (evaluadas en la fase 2 con
`scripts/evaluar_nuevas.py`) como **buena compra**, **mala compra** o **sin
clasificar**, usando el ratio `precio_predicho_usd / precio_publicado_usd` con
zona neutra `1 ± desviación estándar` del ratio sobre el lote de la corrida
(roadmap, fase 3). El score relativo es el propio ratio: ordenado descendente,
las mejores oportunidades aparecen primero.

**Entrada:** `data/processed/propiedades_nuevas_evaluadas.csv` (si no existe,
se genera un dataset evaluado sintético para demostrar el flujo).
**Salida:** `reports/figures/oportunidades_ratios.png` (distribución de ratios).


In [1]:
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.use("Agg")

# Bootstrap para importar `real_estate` (layout src/) sin instalar el paquete.
SRC_DIR = Path.cwd().parent / "src"
if not SRC_DIR.exists():
    SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from real_estate.serving.clasificacion import (  # noqa: E402
    clasificar_oportunidades,
)

RUTA_EVALUADO = Path("../data/processed/propiedades_nuevas_evaluadas.csv")
RUTA_FIGURAS = Path("../reports/figures")

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

## 1. Carga del dataset evaluado

Se carga el CSV evaluado por `scripts/evaluar_nuevas.py`. Si todavía no existe
(primera corrida del pipeline de nuevas publicaciones), se construye un dataset
sintético con la misma estructura para poder mostrar el flujo completo: precios
publicados con señal y predicciones con ruido alrededor del valor razonable.


In [2]:
if RUTA_EVALUADO.exists():
    df = pd.read_csv(RUTA_EVALUADO, low_memory=False)
    print(f"Cargado evaluado real: {df.shape[0]:,} filas x {df.shape[1]} columnas")
else:
    rng = np.random.default_rng(7)
    n = 120
    # Precio publicado en USD con señal de mercado.
    precio_publicado = np.clip(
        rng.uniform(40000, 250000, n) * rng.lognormal(0, 0.35, n),
        15000,
        None,
    )
    # La predicción "acierta" en promedio (ratio ~1) con desvíos que generan
    # buenas y malas compras.
    precio_predicho = precio_publicado * rng.lognormal(0, 0.25, n)
    df = pd.DataFrame(
        {
            "id": [f"nuevo_{i}" for i in range(n)],
            "titulo": [f"Publicación {i}" for i in range(n)],
            "link": [f"https://www.argenprop.com/publicacion-{i}" for i in range(n)],
            "barrio": rng.choice(
                ["Palermo", "Caballito", "Belgrano", "Recoleta", "Villa Crespo"], n
            ),
            "tipo_propiedad": rng.choice(["departamento", "casa", "ph"], n),
            "precio_usd": precio_publicado,
            "precio_predicho_usd": precio_predicho,
            "fecha_prediccion": "2026-08-16",
        }
    )
    print(f"Evaluado real no encontrado; se usó dataset sintético: {len(df):,} filas")

df.head()

Evaluado real no encontrado; se usó dataset sintético: 120 filas


,id,titulo,link,barrio,tipo_propiedad,precio_usd,precio_predicho_usd,fecha_prediccion
0,nuevo_0,Publicación 0,https://www.argenprop.com/publicacion-0,Villa Crespo,ph,183222.746791,127515.337663,2026-08-16
1,nuevo_1,Publicación 1,https://www.argenprop.com/publicacion-1,Caballito,casa,235667.278696,210055.655345,2026-08-16
2,nuevo_2,Publicación 2,https://www.argenprop.com/publicacion-2,Villa Crespo,casa,164979.912950,198665.606636,2026-08-16
3,nuevo_3,Publicación 3,https://www.argenprop.com/publicacion-3,Recoleta,casa,83743.845313,82034.762159,2026-08-16
4,nuevo_4,Publicación 4,https://www.argenprop.com/publicacion-4,Palermo,departamento,51206.002205,52254.204074,2026-08-16


## 2. Clasificación buena / mala compra

Se agregan `ratio_precio` y `clasificacion` con `clasificar_oportunidades`. La
zona neutra es `1 ± std` del ratio sobre el lote (desviación muestral); los
ratios con precio publicado inválido quedan `sin_clasificar` (NaN).


In [3]:
df = clasificar_oportunidades(df)

print("Clasificación del lote:")
print(df["clasificacion"].value_counts().to_string())

print("\nResumen del ratio (score relativo):")
print(df["ratio_precio"].describe().round(3).to_string())

df.head()

Clasificación del lote:
clasificacion
sin_clasificar    83
buena_compra      23
mala_compra       14

Resumen del ratio (score relativo):
count    120.000
mean       1.022
std        0.238
min        0.444
25%        0.869
50%        0.996
75%        1.148
max        1.655


,id,titulo,link,barrio,tipo_propiedad,precio_usd,precio_predicho_usd,fecha_prediccion,ratio_precio,clasificacion
0,nuevo_0,Publicación 0,https://www.argenprop.com/publicacion-0,Villa Crespo,ph,183222.746791,127515.337663,2026-08-16,0.695958,mala_compra
1,nuevo_1,Publicación 1,https://www.argenprop.com/publicacion-1,Caballito,casa,235667.278696,210055.655345,2026-08-16,0.891323,sin_clasificar
2,nuevo_2,Publicación 2,https://www.argenprop.com/publicacion-2,Villa Crespo,casa,164979.912950,198665.606636,2026-08-16,1.204181,sin_clasificar
3,nuevo_3,Publicación 3,https://www.argenprop.com/publicacion-3,Recoleta,casa,83743.845313,82034.762159,2026-08-16,0.979592,sin_clasificar
4,nuevo_4,Publicación 4,https://www.argenprop.com/publicacion-4,Palermo,departamento,51206.002205,52254.204074,2026-08-16,1.020470,sin_clasificar


## 3. Distribución de ratios

Histograma del score relativo con la zona neutra `1 ± std` marcada. Las
publicaciones a la derecha del límite superior son **buenas compras** (el
modelo valora más que lo publicado); a la izquierda del límite inferior,
**malas compras**.


In [4]:
ratios = df["ratio_precio"].dropna()
std = ratios.std()
limite_superior = 1 + std
limite_inferior = 1 - std

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(ratios, bins=25, color="#4c72b0", edgecolor="white", alpha=0.85)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=1, label="ratio = 1")
ax.axvline(
    limite_superior,
    color="green",
    linestyle="--",
    linewidth=1.5,
    label=f"buena compra (> {limite_superior:.2f})",
)
ax.axvline(
    limite_inferior,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label=f"mala compra (< {limite_inferior:.2f})",
)
ax.set_xlabel("ratio = precio_predicho_usd / precio_publicado_usd")
ax.set_ylabel("publicaciones")
ax.set_title(f"Distribución del ratio de oportunidad (std del lote = {std:.3f})")
ax.legend()

RUTA_FIGURAS.mkdir(parents=True, exist_ok=True)
ruta_png = RUTA_FIGURAS / "oportunidades_ratios.png"
fig.savefig(ruta_png, dpi=150, bbox_inches="tight")
print(f"Figura guardada: {ruta_png}")
plt.show()

Figura guardada: ..\reports\figures\oportunidades_ratios.png


C:\Users\Manaus\AppData\Local\Temp\ipykernel_4016\2871916117.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Ranking de oportunidades

Ordenado por `ratio_precio` descendente: la primera fila es la mejor
oportunidad relativa del lote.


In [5]:
columnas = [
    "id",
    "titulo",
    "barrio",
    "tipo_propiedad",
    "precio_usd",
    "precio_predicho_usd",
    "ratio_precio",
    "clasificacion",
]
ranking = df[columnas].sort_values("ratio_precio", ascending=False, na_position="last")
print(f"Top {min(10, len(ranking))} oportunidades (ratio descendente):")
ranking.head(10)

Top 10 oportunidades (ratio descendente):


,id,titulo,barrio,tipo_propiedad,precio_usd,precio_predicho_usd,ratio_precio,clasificacion
90,nuevo_90,Publicación 90,Villa Crespo,ph,33642.179238,55682.149931,1.655129,buena_compra
103,nuevo_103,Publicación 103,Caballito,casa,84379.091871,137466.792685,1.629157,buena_compra
33,nuevo_33,Publicación 33,Belgrano,departamento,66088.097210,101644.514003,1.538015,buena_compra
66,nuevo_66,Publicación 66,Villa Crespo,ph,125496.914659,187346.047226,1.492834,buena_compra
52,nuevo_52,Publicación 52,Recoleta,departamento,61491.463783,91028.698640,1.480347,buena_compra
48,nuevo_48,Publicación 48,Villa Crespo,departamento,189644.554007,277936.883788,1.465567,buena_compra
83,nuevo_83,Publicación 83,Villa Crespo,ph,92620.256243,135587.505758,1.463908,buena_compra
100,nuevo_100,Publicación 100,Caballito,ph,143130.168397,209525.193117,1.463879,buena_compra
61,nuevo_61,Publicación 61,Recoleta,casa,149087.801170,217563.939032,1.459301,buena_compra
108,nuevo_108,Publicación 108,Palermo,casa,91053.669310,131793.349008,1.447425,buena_compra
